# Mastra + MongoDB Atlas: 3. Full-Text, Hybrid Search & Reranking

Notebook 01 built semantic search and it worked well. This notebook starts by breaking it.

Vector search matches meaning, which is exactly what you want until the user types a
proper noun. Ask `sample_mflix` for **Tatooine** and semantic search leads with *Dune*
(a desert planet, reasonably enough) and puts a Star Wars film second. The embedding does
not know that word names a specific place; it only knows the neighbourhood it lives in, so
it cannot tell you whether it matched or guessed.

Full-text search has the opposite problem and the opposite strength. `$rankFusion` runs
both and fuses the rankings on the server.

## What you'll learn
1. Where semantic search fails, with three queries that demonstrate it
2. `textQuery`: BM25 full-text search, and why a BYO index needs to be told where to look
3. `hybridQuery`: server-side `$rankFusion`, and exactly how its scores are computed
4. `weights`: what tilting toward vector or text actually does to the ranking
5. Reranking with `rerank-2.5`: a cross-encoder as a final pass

## Prerequisites
- `sample_mflix` loaded, and the setup from notebook 01 (`.env`, `npm install`)
- **MongoDB 8.0 or later** for `$rankFusion` (see the note in section 3)

Notebooks 01 and 02 are the natural reading order, but not a hard requirement: the setup
cell below creates both Atlas indexes this notebook needs if they are not already there.

In [ ]:
import { MongoDBVector } from '@mastra/mongodb';

import { loadNotebookConfig } from './src/config.js';
import {
  APP_NAME,
  MFLIX_FULLTEXT_INDEX,
  MFLIX_VECTOR_INDEX,
  connect,
  createAtlasEmbedder,
  ensureFullTextSearchIndex,
  ensureVectorSearchIndex,
} from './src/utils.js';

const cfg = loadNotebookConfig();
console.log('Configuration:', cfg.summary);

const vector = new MongoDBVector({
  id: 'mflix',
  uri: cfg.mongodbUri,
  dbName: cfg.dbName,
  options: { appName: APP_NAME },
  embeddingFieldPath: 'plot_embedding_voyage_3_large',
});
await vector.connect();

// Both Atlas indexes this notebook queries, created if absent. Notebook 01 makes the
// vector one, so on the happy path both of these are no-ops -- but running this
// notebook on its own works too, which is the point.
const { client } = await connect(cfg.mongodbUri);
const db = client.db(cfg.dbName);
const movies = db.collection('embedded_movies');

// Passing `db` turns on the index-budget preflight: at the cluster's search index
// limit these stop and list what is using the slots, rather than removing anything.
await ensureVectorSearchIndex(
  movies,
  {
    indexName: MFLIX_VECTOR_INDEX,
    path: 'plot_embedding_voyage_3_large',
    dimensions: cfg.embeddingDimensions,
    filterPaths: ['genres', 'year'],
  },
  // `db: db` rather than the `{ db }` shorthand: the tslab kernel does not resolve a
  // shorthand property against a variable declared in an earlier cell.
  { db: db },
);
// Full-text over `plot`. Section 2 explains why it has to be named explicitly
// on a bring-your-own registration.
const TEXT_INDEX = await ensureFullTextSearchIndex(
  movies,
  { indexName: MFLIX_FULLTEXT_INDEX, fields: ['plot'] },
  // `db: db` rather than the `{ db }` shorthand: the tslab kernel does not resolve a
  // shorthand property against a variable declared in an earlier cell.
  { db: db },
);

const INDEX = 'mflix_plots';
await vector.createIndex({
  indexName: INDEX,
  dimension: cfg.embeddingDimensions,
  metric: 'cosine',
  collectionName: 'embedded_movies',
  searchIndexName: MFLIX_VECTOR_INDEX,
  filterFields: ['genres', 'year'],
});

const embedder = createAtlasEmbedder({
  apiKey: cfg.atlasApiKey,
  model: cfg.embeddingModel,
  dimensions: cfg.embeddingDimensions,
});

console.log('Ready. Logical index:', INDEX);

Configuration: db=sample_mflix · embeddings=voyage-3-large (2048-d) · llm=claude-haiku-4-5 via gateway


  movies_vector_index already exists.


  movies_fulltext_index already exists.


Ready. Logical index: mflix_plots


## 1. Where vector search falls down

Three queries, each a single proper noun that a plot summary would contain verbatim.
`titlesOf` keeps the output readable across the comparisons that follow.

In [ ]:
interface Movie { title?: string; year?: number; plot?: string; genres?: string[] }

/** Format results as a one-line, ranked list of titles. */
function titlesOf(results: Array<{ metadata?: unknown }>): string {
  if (results.length === 0) return '(no results)';
  return results.map((hit, i) => `${i + 1}. ${(hit.metadata as Movie).title}`).join('   ');
}

for (const query of ['Tatooine', 'xenomorph', 'Vito Corleone']) {
  const semantic = await vector.query({
    indexName: INDEX,
    queryVector: await embedder.embedQuery(query),
    topK: 3,
    metadataMode: 'document',
  });
  console.log(`"${query}"`);
  console.log(`   vector: ${titlesOf(semantic)}\n`);
}

"Tatooine"


   vector: 1. Dune   2. Star Wars: The Clone Wars   3. Message from Space



"xenomorph"


   vector: 1. Hardware   2. Doom   3. Aliens



"Vito Corleone"


   vector: 1. Scarface   2. A Brooklyn State of Mind   3. The Way of the Dragon



These are not absurd answers. *Dune* really is the nearest thing to Tatooine in
embedding space, and *Scarface* is a reasonable neighbour for a mafia patriarch. They are
just not what the user asked for. Someone who types "Tatooine" wants Star Wars, and
someone who types "Vito Corleone" wants *The Godfather*.

This is the characteristic failure of pure vector search, and it is worse than a wrong
answer because the results look plausible. Nothing in the response signals that the query
contained a term the model could only approximate.

## 2. Full-text search

`textQuery` runs a BM25 `$search` against an Atlas Search index. It matches terms, not
meaning: no embedding is involved, and the query is not embedded.

One thing to get right first. On a bring-your-own index (which this is, from notebook 02),
Mastra does **not** guess the full-text index name. A managed index gets a companion
text index automatically, so its name is known. A BYO collection may have several text
indexes with different mappings, or none, so guessing would mean silently querying the
wrong one. Instead it asks.

In [ ]:
try {
  await vector.textQuery({ indexName: INDEX, query: 'Tatooine', paths: ['plot'], topK: 3 });
} catch (error) {
  console.log((error as Error).message.split('\n')[0]);
}

No full-text search index exists for bring-your-own index "mflix_plots". Call createSearchIndex({ indexName: "mflix_plots" }) to enable textQuery/hybridQuery, or pass an explicit searchIndexName.


The error names both ways forward: `createSearchIndex` to have Mastra provision and
remember a text index, or `searchIndexName` to point at one that already exists. We use
the second, because `movies_fulltext_index` is already there, and on a free M0 cluster
with a three-index budget, provisioning a fourth would fail anyway.

Now the same three queries, through BM25.

In [ ]:
for (const query of ['Tatooine', 'xenomorph', 'Vito Corleone']) {
  const fullText = await vector.textQuery({
    indexName: INDEX,
    query,
    paths: ['plot'],              // the field the index maps
    topK: 3,
    metadataMode: 'document',
    searchIndexName: TEXT_INDEX,  // BYO: say which index explicitly
  });
  console.log(`"${query}"`);
  console.log(`   text:   ${titlesOf(fullText)}\n`);
}

"Tatooine"


   text:   (no results)



"xenomorph"


   text:   (no results)



"Vito Corleone"


   text:   (no results)



All three return nothing.

That is not a failure of BM25; it is the truth about this dataset. `sample_mflix` stores
one-sentence plot summaries, and none of them contain the words "Tatooine", "xenomorph",
or "Vito Corleone". Full-text search cannot match a term that is not in the corpus, and it
says so by returning empty rather than by returning something adjacent.

Which reframes the whole comparison. Vector search returning *Dune* for "Tatooine" is
better than nothing; it is a reasonable guess in the absence of a literal match. The
problem was never that vector search is bad here; it is that you cannot tell from its
output whether it matched or guessed.

So take a query where the term *is* in the corpus, and the two disagree properly.

In [ ]:
const query = 'the Millennium Falcon';
const queryVector = await embedder.embedQuery(query);

const semantic = await vector.query({
  indexName: INDEX, queryVector: queryVector, topK: 5, metadataMode: 'document',
});
const fullText = await vector.textQuery({
  indexName: INDEX, query: query, paths: ['plot'], topK: 5,
  metadataMode: 'document', searchIndexName: TEXT_INDEX,
});

console.log(`"${query}"\n`);
console.log('vector:', titlesOf(semantic));
console.log('text:  ', titlesOf(fullText));

"the Millennium Falcon"



vector: 1. Message from Space   2. Message from Space   3. Starcrash   4. Star Wars: Episode IV - A New Hope   5. Star Wars: Episode VI - Return of the Jedi


text:   1. After Earth   2. Ewoks: The Battle for Endor   3. Bionicle: Mask of Light   4. We Were Soldiers   5. Titanic


Both are unsatisfying, in opposite directions.

Vector search leads with the *genre*, 1970s and 80s space opera, and *Message from Space*
twice, because the dataset contains duplicate documents. It does reach the real Star Wars
films, but only at positions 4 and 5, below three films that merely feel similar.

Full-text search matches the individual words. "Falcon" and "Millennium" appear in plots
that have nothing to do with each other, so it returns *After Earth* and an Ewoks film:
term overlap without topical relevance.

Neither ranking puts a Star Wars film first, and each has partial signal the other lacks.
That is the situation `$rankFusion` is built for.

## 3. Hybrid search with `$rankFusion`

`hybridQuery` sends one aggregation containing both searches and fuses their rankings
**on the server**: one round trip, no client-side merging.

> **Version note.** `$rankFusion` requires MongoDB 8.0+. It is generally available from
> 8.1; on 8.0.x it may need to be enabled via a MongoDB support case, though it runs on
> Atlas 8.0.x clusters. Mastra gates at >= 8.0 and lets the server be the authority. If
> the stage is not enabled, the query itself says so.

In [ ]:
const hybrid = await vector.hybridQuery({
  indexName: INDEX,
  queryVector: queryVector,   // the vector leg
  query: query,               // the text leg
  paths: ['plot'],
  topK: 5,
  metadataMode: 'document',
  textSearchIndexName: TEXT_INDEX,
});

console.log(`"${query}"\n`);
console.log('vector:', titlesOf(semantic));
console.log('text:  ', titlesOf(fullText));
console.log('hybrid:', titlesOf(hybrid));

"the Millennium Falcon"



vector: 1. Message from Space   2. Message from Space   3. Starcrash   4. Star Wars: Episode IV - A New Hope   5. Star Wars: Episode VI - Return of the Jedi


text:   1. After Earth   2. Ewoks: The Battle for Endor   3. Bionicle: Mask of Light   4. We Were Soldiers   5. Titanic


hybrid: 1. Star Wars: Episode VI - Return of the Jedi   2. Message from Space   3. Message from Space   4. After Earth   5. Ewoks: The Battle for Endor


*Return of the Jedi* is now first. It was fifth on the vector leg and sixteenth on the text
leg: top of neither, but present in both.

That is the point of rank fusion. A document ranked moderately well by **both** retrievers
beats a document ranked highly by only one. The films that beat it on similarity alone
(*Message from Space*, *Starcrash*) contribute nothing from the text leg, and the films BM25
liked (*After Earth*, the Ewoks film) contribute nothing from the vector leg. Only Star Wars
scored on both, so the fusion promoted it above both sets of specialists.

### How the score is computed

Worth being precise about, because the numbers look arbitrary (around 0.03), and they are
not.

`$rankFusion` uses **Reciprocal Rank Fusion**. Each retriever contributes `1 / (k + rank)`
for each document it returned, with `k = 60`, and the contributions are summed. Only
*ranks* matter; the cosine similarity and the BM25 score never meet, which is what makes
fusing them legitimate: they are not comparable quantities, but their orderings are.

So the scores should be reconstructable from two separate queries. Mostly, they are.

In [ ]:
// Rank each document under each retriever, over a deeper pool than we display.
const deepVector = await vector.query({
  indexName: INDEX, queryVector: queryVector, topK: 20, metadataMode: 'document',
});
const deepText = await vector.textQuery({
  indexName: INDEX, query: query, paths: ['plot'], topK: 20,
  metadataMode: 'document', searchIndexName: TEXT_INDEX,
});

const vectorRank = new Map(deepVector.map((hit, i) => [hit.id, i + 1]));
const textRank = new Map(deepText.map((hit, i) => [hit.id, i + 1]));

const K = 60;
console.log('title                              vRank  tRank   1/(60+v) + 1/(60+t)   actual');
for (const hit of hybrid) {
  const v = vectorRank.get(hit.id);
  const t = textRank.get(hit.id);
  const predicted = (v ? 1 / (K + v) : 0) + (t ? 1 / (K + t) : 0);
  console.log(
    `${String((hit.metadata as Movie).title).slice(0, 34).padEnd(34)} ` +
    `${String(v ?? '-').padStart(5)}  ${String(t ?? '-').padStart(5)}   ` +
    `${predicted.toFixed(6).padStart(19)}   ${hit.score.toFixed(6)}`,
  );
}

title                              vRank  tRank   1/(60+v) + 1/(60+t)   actual


Star Wars: Episode VI - Return of      5     16              0.028543   0.028898


Message from Space                     2      -              0.016129   0.016393


Message from Space                     1      -              0.016393   0.016393


After Earth                            -      1              0.016393   0.016393


Ewoks: The Battle for Endor            -      2              0.016129   0.016129


Most rows land on the actual score exactly, and the rest land close: `0.028543` predicted
against `0.028898` actual, for instance. The near misses are informative, and they are not
a flaw in the formula.

`$rankFusion` ranks each leg over the candidate pool *it* assembles inside the aggregation.
The two probe queries above assemble their own pools at `topK: 20`, so a rank measured
client-side is only an estimate of the rank the server used. Probing deeper does not fix
this. It can make agreement worse, because a document that entered our pool at position 10
may have been at position 3 in the server's. Treat this table as evidence for the *shape* of
the formula rather than as an exact audit; if you need the true per-leg ranks, read them from
`explain` on the aggregation rather than reconstructing them.

Two consequences worth knowing:

- **The scores are not similarities.** A hybrid score near 0.03 is not "3% relevant". The
  theoretical maximum for two retrievers is `2/61 ≈ 0.0328`, reached only by a document
  ranked first in both. Compare hybrid scores with each other, never against a cosine
  similarity from `query`.
- **A document only one retriever found is not excluded**, it just contributes one term
  instead of two. Vector-only hits still appear, ranked below documents both retrievers
  agreed on.

## 4. Weights

`weights` scales each retriever's contribution: the score becomes
`w_vector/(k + rank_vector) + w_text/(k + rank_text)`. Default is `1` and `1`.

Note that `{vector: 0.5, text: 0.5}` is **not** the default. It halves every score and
changes no ordering. What matters is the *ratio*.

In [ ]:
const submarine = 'submarine cat and mouse';
const submarineVector = await embedder.embedQuery(submarine);

console.log(`"${submarine}"\n`);
for (const weights of [
  undefined,
  { vector: 0.5, text: 0.5 },
  { vector: 0.9, text: 0.1 },
  { vector: 0.1, text: 0.9 },
]) {
  const results = await vector.hybridQuery({
    indexName: INDEX,
    queryVector: submarineVector,
    query: submarine,
    paths: ['plot'],
    topK: 3,
    metadataMode: 'document',
    textSearchIndexName: TEXT_INDEX,
    ...(weights ? { weights: weights } : {}),
  });
  const label = weights ? `v${weights.vector} / t${weights.text}` : 'default (1 / 1)';
  console.log(`${label.padEnd(16)} ${titlesOf(results)}`);
  console.log(`${''.padEnd(16)} top score ${results[0]?.score.toFixed(6)}`);
}

"submarine cat and mouse"



default (1 / 1)  1. U-571   2. Men Without Women   3. The Hunt for Red October


                 top score 0.030478


v0.5 / t0.5      1. U-571   2. Men Without Women   3. The Hunt for Red October


                 top score 0.015239


v0.9 / t0.1      1. U-571   2. The Hunt for Red October   3. Men Without Women


                 top score 0.016163


v0.1 / t0.9      1. Running Out of Time 2   2. Men Without Women   3. Julayi


                 top score 0.014754


The second row makes the first point. `0.5/0.5` returns the same order as the default with
every score exactly halved. The ratio is unchanged, so the ranking is unchanged. Passing it
because it looks like "balanced" accomplishes nothing.

The last row makes the second. For "submarine cat and mouse" the phrase is idiomatic: no
plot says "cat and mouse" about a submarine, so the meaning is carried almost entirely by
the vector leg. Tilt to `0.1/0.9` and the submarine films fall out of the top three
altogether, replaced by whatever matched "cat" and "mouse" literally. Tilting the other way,
`0.9/0.1`, only shuffles the same three films, because the vector leg already dominated the
default ranking, so removing text influence has little left to change.

A rule of thumb worth holding loosely: **weight text up for queries full of identifiers**
(SKUs, error codes, names, versions), **weight vector up for descriptive or idiomatic
queries**. If your traffic is a mix, the default is a reasonable neutral, and this is a
parameter to tune against your own relevance judgements rather than by intuition.

`filter` works on `hybridQuery` too, applied to both legs.

In [ ]:
const modern = await vector.hybridQuery({
  indexName: INDEX,
  queryVector: submarineVector,
  query: submarine,
  paths: ['plot'],
  topK: 3,
  filter: { year: { $gte: 1990 } },   // declared filter field -> pushed down
  metadataMode: 'document',
  textSearchIndexName: TEXT_INDEX,
});

for (const hit of modern) {
  const movie = hit.metadata as Movie;
  console.log(`${hit.score.toFixed(6)}  ${String(movie.title).padEnd(28)} ${movie.year}`);
}

0.031099  U-571                        2000


0.029857  The Rift                     1990


0.029762  The Hunt for Red October     1990


## 5. Reranking

Hybrid search fixes *which* documents come back. Reranking fixes their *order*.

The reason a second pass helps is architectural. Vector search compares two embeddings
computed independently. The query never sees the document. That is what makes it fast
enough to search millions of documents, and it is also what limits its precision.

A **cross-encoder** reads the query and one document *together* and scores that pair. Far
more accurate, far too slow to run over a whole collection. So: retrieve broadly with
hybrid search, then rerank the top few dozen. Voyage's `rerank-2.5` is available through
the same Atlas endpoint, wrapped as `atlasRerank` in `src/utils.ts`.

In [ ]:
import { atlasRerank } from './src/utils.js';

const rerankQuery = 'a man loses his memory and discovers he was an assassin';

// Retrieve broadly -- 25 candidates, more than we intend to show.
const candidates = await vector.hybridQuery({
  indexName: INDEX,
  queryVector: await embedder.embedQuery(rerankQuery),
  query: rerankQuery,
  paths: ['plot'],
  topK: 25,
  metadataMode: 'document',
  textSearchIndexName: TEXT_INDEX,
});

// Rerank on the plot text, keeping the mapping back to the movies.
const reranked = await atlasRerank(
  rerankQuery,
  candidates.map((hit) => String((hit.metadata as Movie).plot ?? '')),
  { apiKey: cfg.atlasApiKey, topN: 5 },
);

console.log(`"${rerankQuery}"\n`);
console.log('hybrid top 5:');
for (const hit of candidates.slice(0, 5)) {
  console.log(`   ${hit.score.toFixed(6)}  ${(hit.metadata as Movie).title}`);
}

console.log('\nafter reranking:');
for (const result of reranked) {
  const movie = candidates[result.index]!.metadata as Movie;
  // `index` maps back to the array we passed in, which is why the order survives.
  console.log(`   ${result.score.toFixed(4)}  ${String(movie.title).padEnd(30)} (was #${result.index + 1})`);
}

"a man loses his memory and discovers he was an assassin"



hybrid top 5:


   0.032002  Who Am I?


   0.030550  Ghajini


   0.030214  The Bourne Identity


...

   0.7578  The Bourne Supremacy           (was #17)


   0.7031  The Bourne Identity            (was #3)


   0.6328  Who Am I?                      (was #1)


   0.5742  Wanted                         (was #5)


   0.5586  Imaginaerum                    (was #21)


The `(was #N)` column is where to look: it shows how far each film moved. This query is a
clean win: a Bourne film was sitting deep in the candidate list and the cross-encoder
pulled it to the top, which is exactly the pattern reranking is supposed to produce.

It is worth being honest that this does not happen on every query. We tried six against
this collection and the results were genuinely mixed. "a submarine crew hunts an enemy
vessel" promoted *Run Silent Run Deep* from #11 to #1. But "a tense standoff between a
detective and a professional thief" got *worse*: reranking dropped *Heat* and *Entrapment*
in favour of lighter heist films, and "two rival magicians destroy each other with
obsession" surfaced *Prospero's Books* at 0.40.

The reason is the corpus, not the model. `sample_mflix` plots are one sentence. A
cross-encoder earns its cost by reading a document closely, and there is almost nothing
here to read, so on short text it sometimes latches onto surface phrasing that the first
pass had correctly discounted. Expect reranking to pay off on paragraphs and documents, and
to be worth measuring rather than assuming on anything shorter.

Two things about the scores. They are relevance scores in 0 to 1 and are comparable *to each
other*, unlike RRF scores. And `atlasRerank` returns `index`, the position in the array
you passed in, which is what lets you map back to your own objects rather than to the
strings you sent.

The cost is a second network call and the latency of scoring every candidate. Reranking 25
candidates is cheap; reranking 500 is not. `topK` on the retrieval step is the dial.

## 6. All three, side by side

Ending with the comparison that motivated the notebook: one query, four strategies.

In [ ]:
/** Run all four strategies for `text` and print their rankings together. */
async function compare(text: string, topK = 4): Promise<void> {
  const vec = await embedder.embedQuery(text);

  const [semanticOnly, textOnly, fused] = await Promise.all([
    vector.query({ indexName: INDEX, queryVector: vec, topK: topK, metadataMode: 'document' }),
    vector.textQuery({
      indexName: INDEX, query: text, paths: ['plot'], topK: topK,
      metadataMode: 'document', searchIndexName: TEXT_INDEX,
    }),
    vector.hybridQuery({
      indexName: INDEX, queryVector: vec, query: text, paths: ['plot'], topK: 25,
      metadataMode: 'document', textSearchIndexName: TEXT_INDEX,
    }),
  ]);

  const ranked = await atlasRerank(
    text,
    fused.map((hit) => String((hit.metadata as Movie).plot ?? '')),
    { apiKey: cfg.atlasApiKey, topN: topK },
  );

  console.log(`"${text}"`);
  console.log('  vector  :', titlesOf(semanticOnly));
  console.log('  text    :', titlesOf(textOnly));
  console.log('  hybrid  :', titlesOf(fused.slice(0, topK)));
  console.log('  reranked:', ranked.map((r, i) => `${i + 1}. ${(fused[r.index]!.metadata as Movie).title}`).join('   '));
  console.log();
}

await compare('a hacker discovers reality is a simulation');
await compare('Hogwarts');

"a hacker discovers reality is a simulation"


  vector  : 1. The Matrix   2. Virtuosity   3. TRON   4. Gamer


  text    : 1. The Matrix   2. Avalon   3. Swordfish   4. TRON


  hybrid  : 1. The Matrix   2. TRON   3. Virtuosity   4. Arrambam


  reranked: 1. The Matrix   2. TRON   3. Spy Kids 3-D: Game Over   4. Arrambam


"Hogwarts"


  vector  : 1. Harry Potter and the Sorcerer's Stone   2. Harry Potter and the Chamber of Secrets   3. Harry Potter and the Order of the Phoenix   4. Harry Potter and the Prisoner of Azkaban


  text    : 1. Harry Potter and the Deathly Hallows: Part 2   2. Harry Potter and the Chamber of Secrets   3. Harry Potter and the Order of the Phoenix   4. Harry Potter and the Sorcerer's Stone


  hybrid  : 1. Harry Potter and the Chamber of Secrets   2. Harry Potter and the Sorcerer's Stone   3. Harry Potter and the Order of the Phoenix   4. Harry Potter and the Deathly Hallows: Part 2


  reranked: 1. Harry Potter and the Sorcerer's Stone   2. Harry Potter and the Prisoner of Azkaban   3. Harry Potter and the Chamber of Secrets   4. Harry Potter and the Order of the Phoenix


Both queries land on the right film, by different routes. The simulation query is
descriptive, so the vector leg carries it: *The Matrix* first everywhere, and the reranker
trades *Virtuosity* for *Spy Kids 3-D* at position three, which is the short-plot problem
again. "Hogwarts" is a proper noun that *is* in the corpus, so both legs have real signal and
each of the four strategies picks a different Harry Potter film to lead with.

That last detail is the honest summary of the notebook. Every strategy here is defensible on
this query, and choosing between them is a relevance-measurement question, not something you
can settle by reading about `$rankFusion`. What hybrid buys you is not a better answer on
every query; it is not having to know in advance which leg a given query needs.

## Housekeeping

**Created by this notebook:** `movies_fulltext_index` on `embedded_movies`, if it was not
already there, plus the same `movies_vector_index` and `mflix_plots` registration
notebook 01 makes -- created here too, so this notebook runs on its own.

**Deleted by this notebook:** nothing. It writes no documents to `sample_mflix` and
removes none, and it drops no collection and no index.

**Left in place, deliberately:** both Atlas indexes and the registration. Notebook 04
queries all three, and the LangChain set uses both index names.

**Not called:** `vector.deleteIndex`, for the reason notebook 01 gives -- on a
bring-your-own registration it drops the Atlas index the registration points at, and
`movies_vector_index` is shared.

**To remove what this notebook added**, drop `movies_fulltext_index` (and
`movies_vector_index`, if you are done with the whole set) from the Atlas UI.

In [ ]:
await vector.disconnect();
await client.close();
console.log('Disconnected. Both Atlas indexes untouched.');

Disconnected. Both Atlas indexes untouched.


## Recap

You saw vector search return confident, plausible, wrong answers for proper nouns; saw
full-text search return nothing rather than guess; and used `$rankFusion` to get one
ranking out of both in a single round trip.

Worth carrying forward:

- **A BYO index will not guess its text index.** Pass `searchIndexName` /
  `textSearchIndexName`, or register one with `createSearchIndex`.
- **RRF scores are not similarities.** `1/(60+rank)` summed per retriever, max `≈0.0328`
  for two legs. Compare them only to each other.
- **`weights` is a ratio.** `0.5/0.5` changes nothing but the scale.
- **Rerank a broad candidate set, not the collection.** Retrieve 25, rerank, show 5. And
  measure it. On one-sentence plots reranking helped on some queries here and hurt on
  others, because a cross-encoder needs text to read.

**Next:** `04_agent_with_tools_and_memory.ipynb` puts this behind a Mastra `Agent`:
retrieval as a tool the model decides when to call, and conversation history persisted in
MongoDB so it survives a restart.